# Error predictor, per gene: mean alone vs mean + uncertainty (rebuttal, reported version)

The earlier pooled version (one forest over all genes, rank-correlation score) is `error_predictor_pooled_over_genes.ipynb`; it is superseded by this one.

Reads `<run-dir>/error_predictor_gene/fold{j}.parquet` from `snake/scripts/canals/error_predictor_gene_precompute.py` (one job per held-out donor, launcher `launch_error_predictor_precompute.sh`).

- Per gene, fitted on fold j's network errors on its validation donor: forest A `|err| ~ pct(mu)`, forest B `|err| ~ pct(mu) + pct(u)`. Leaf size tuned on an 80/20 split of the validation donor's spots, then refitted on all of them. The held-out donor is only scored.
- Score = Table 4's withholding share, same cascade as the paper's withholding table: per gene its slides' curves n-weighted (all held-out donors pooled), mean over genes, AUSE = mean over 19 fractions, share = (1 - AUSE) / (1 - AUSE oracle). Ties of the forests' outputs are withheld proportionally (exact expectation); without ties this is canal 2's curve.
- Spread: the same share per held-out donor, B - A counted by sign. Second score: Spearman, donor cascade as the correlation figures.
- Sanity: Poisson B - A = 0; u alone reproduces Table 4 (`canal2_uncertainty/sparsification.parquet`).

In [3]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))
RUN_ID = "pr_classic3"
RESULTS = PROJ / "results"
DATASETS = ["10x_TuPro", "TLS", "LUNG_CANCER"]
RADIUS = "r75"
LAMBDA_DIR = "lambda_0.0"  # Gaussian heads at lambda 0.0
SUBDIR = "error_predictor_gene"
MODELS = ["heteroscedastic", "spot_homoscedastic", "negativebinomial_hetero_noanchor_cp",
          "poisson_cp", "homoscedastic"]      # last two = sanity heads
CASES = ["lin_all", "lin_pos"]
N_PLUS_MIN = 60
NQ = 19                                # fractions 0, 0.05, ..., 0.9


def ause(t, s):
    """Table 4's cascade on the (gene, slide) rows of t: per gene n-weighted, mean over genes, mean over fractions."""
    cols = [f"c_{s}_{q:02d}" for q in range(NQ)]
    per_gene = t[cols].mul(t["n"], axis=0).groupby(t["gene"]).sum().div(t.groupby("gene")["n"].sum(), axis=0)
    return float(per_gene.mean(axis=0).mean())


def share(t, s):
    return (1 - ause(t, s)) / (1 - ause(t, "oracle"))


def canal2_share(leaf, case, score):
    """Table 4's number from the canal 2 artifact (within (gene, slide)), or NaN if absent."""
    f = leaf / "canal2_uncertainty" / "sparsification.parquet"
    if not f.is_file():
        return np.nan
    sp = pd.read_parquet(f)
    a = lambda x: float(np.mean(sp[x] / sp[x].iloc[0]))
    col = lambda x: f"curve_{x}_gene_slide_{case}"
    if col(score) not in sp.columns:
        return np.nan
    return (1 - a(col(score))) / (1 - a(col("oracle")))


def donor_value(t, col):
    """Spearman donor point: per gene n-weighted over the donor's slides, mean over genes."""
    d = t[t[col].notna()]
    g = d.assign(w=d[col] * d["n"]).groupby("gene")[["w", "n"]].sum()
    return float((g["w"] / g["n"]).mean())


rows, per_donor, leaves = [], [], []
for ds in DATASETS:
    for m in MODELS:
        leaf = RESULTS / ds / RUN_ID / f"_{ds}_hvg_top3000" / RADIUS / m
        if (leaf / LAMBDA_DIR).is_dir():
            leaf = leaf / LAMBDA_DIR
        files = sorted((leaf / SUBDIR).glob("fold*.parquet"))
        if not files:
            print(f"MISSING: {ds} {m}")
            continue
        k_expected = len(json.loads((leaf / "splits.json").read_text())["folds"])
        frames = []
        for f in files:
            s = json.loads(f.with_suffix(".json").read_text())
            frames.append(pd.read_parquet(f).assign(donor=s["test_donor"], val_donor=s["val_donor"]))
            for case in s["cases"]:
                leaves.append(dict(dataset=ds, model=m, case=case, donor=s["test_donor"],
                                   leaf_A=s["best"][case]["A"]["leaf"], leaf_B=s["best"][case]["B"]["leaf"],
                                   genes_fitted=s["per_case"][case]["n_genes_fitted"]))
        T = pd.concat(frames, ignore_index=True)
        T = T[T["n"] >= N_PLUS_MIN]
        for case in CASES:
            tc = T[T["case"] == case]
            tf = tc[tc["fitted"] & tc["c_A_00"].notna() & tc["c_B_00"].notna()]   # ONE population for A, B, refs
            rec = dict(dataset=ds, model=m, case=case, K=len(files), K_expected=k_expected)
            for sc_ in ("A", "B", "u", "mu_high", "mu_low"):
                rec[sc_] = share(tf, sc_)
            rec["B-A"] = rec["B"] - rec["A"]
            rec["mu*"] = max(rec["mu_high"], rec["mu_low"])
            # Table 4 check: u on ALL groups (fitted or not) vs the canal 2 artifact.
            rec["u_all"] = share(tc[tc["c_u_00"].notna()], "u") if tc["c_u_00"].notna().any() else np.nan
            rec["u_tab4"] = canal2_share(leaf, case, "pred")
            diffs, sp_diffs = [], []
            for donor, td in tf.groupby("donor"):
                a, b = share(td, "A"), share(td, "B")
                ra, rb = donor_value(td, "rho_A"), donor_value(td, "rho_B")
                diffs.append(b - a)
                sp_diffs.append(rb - ra)
                per_donor.append(dict(dataset=ds, model=m, case=case, donor=donor,
                                      val_donor=td["val_donor"].iloc[0], A=a, B=b, diff=b - a,
                                      rho_A=ra, rho_B=rb, rho_diff=rb - ra))
            diffs, sp_diffs = np.array(diffs), np.array(sp_diffs)
            rec["B>A donors"] = f"{int((diffs > 1e-4).sum())}/{len(diffs)}"
            rec["max|B-A| donor"] = float(np.abs(diffs).max())
            rec["rho_A"] = float(np.mean([r["rho_A"] for r in per_donor[-len(diffs):]]))
            rec["rho_B-A"] = float(sp_diffs.mean())
            rec["rho B>A donors"] = f"{int((sp_diffs > 1e-4).sum())}/{len(sp_diffs)}"
            rows.append(rec)

RES = pd.DataFrame(rows)
PD = pd.DataFrame(per_donor)
LV = pd.DataFrame(leaves)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 500)
fmt = lambda v: f"{v:+.4f}"

for case in CASES:
    print("=" * 120)
    print(f"{case}: share of the oracle's error reduction, within (gene, slide), all held-out donors pooled (Table 4 cascade)")
    print("  A = forest on mu, B = forest on mu + u; u, mu* = Table 4's columns on the same groups;")
    print("  u_all = u on all groups, u_tab4 = from canal 2 (must match u_all); B>A donors = per-donor share B - A > 1e-4")
    print("=" * 120)
    cols = ["dataset", "model", "K", "K_expected", "A", "B", "B-A", "B>A donors", "max|B-A| donor",
            "u", "mu*", "u_all", "u_tab4", "rho_A", "rho_B-A", "rho B>A donors"]
    print(RES[RES["case"] == case][cols].to_string(index=False, float_format=fmt))

print("\nPer held-out donor (forests fitted on val_donor):")
print(PD.to_string(index=False, float_format=fmt))
print("\nChosen leaf sizes (tuned on the validation donor's 80/20 split) and genes fitted:")
print(LV.to_string(index=False))

lin_all: share of the oracle's error reduction, within (gene, slide), all held-out donors pooled (Table 4 cascade)
  A = forest on mu, B = forest on mu + u; u, mu* = Table 4's columns on the same groups;
  u_all = u on all groups, u_tab4 = from canal 2 (must match u_all); B>A donors = per-donor share B - A > 1e-4
    dataset                               model  K  K_expected       A       B     B-A B>A donors  max|B-A| donor       u     mu*   u_all  u_tab4   rho_A  rho_B-A rho B>A donors
  10x_TuPro                     heteroscedastic  7           7 +0.2241 +0.2392 +0.0151        7/7         +0.0326 +0.2144 +0.1971 +0.2144 +0.2144 +0.2948  +0.0059            5/7
  10x_TuPro                  spot_homoscedastic  7           7 +0.2359 +0.2492 +0.0133        6/7         +0.0296 +0.1204 +0.1907 +0.1204 +0.1204 +0.2945  +0.0037            5/7
  10x_TuPro negativebinomial_hetero_noanchor_cp  7           7 +0.2968 +0.3052 +0.0084        6/7         +0.0211 +0.3262 +0.3493 +0.3262 +0.3262 +0.37

In [4]:
# ── Story table: is the gain of B over A real, and does B beat the best MEAN-ONLY score? ──
# Reads RES / PD from the cell above, nothing recomputed from disk.
#   B-A mean [CI]   per-donor share B - A, mean + Student-t 95% CI over held-out donors (as the
#                   paper's donor intervals; optimistic, folds share training donors)
#   best mean-only  max(A, mu*): the forest on mu, or the raw mean ranking in its better direction.
#                   mu* picks its direction ON THE TEST DATA, so this baseline is optimistic and
#                   B - best mean-only is a conservative test for B.
#   best            which of mu*, u, A, B has the highest share (u alone included).
# The three heads whose u can vary within a (gene, slide); the sanity heads are in the cell above.
from src.evaluation.benchmark_summary import donor_lopo_stats

HEADS = ["heteroscedastic", "spot_homoscedastic", "negativebinomial_hetero_noanchor_cp"]
story = []
for case in CASES:
    for ds in DATASETS:
        for m in HEADS:
            g = PD[(PD["dataset"] == ds) & (PD["model"] == m) & (PD["case"] == case)]
            r = RES[(RES["dataset"] == ds) & (RES["model"] == m) & (RES["case"] == case)]
            if g.empty or r.empty:
                continue
            r = r.iloc[0]
            st = donor_lopo_stats(dict(zip(g["donor"], g["diff"])))
            cands = {"mu*": r["mu*"], "u": r["u"], "A": r["A"], "B": r["B"]}
            story.append(dict(case=case, dataset=ds, model=m, **cands,
                              **{"B-A mean": st["mean"],
                                 "B-A CI": f"[{st['ci95_mean'][0]:+.3f}, {st['ci95_mean'][1]:+.3f}]",
                                 "B>A": f"{int((g['diff'] > 1e-4).sum())}/{len(g)}",
                                 "B - best mean-only": r["B"] - max(r["A"], r["mu*"]),
                                 "best": max(cands, key=cands.get)}))
STORY = pd.DataFrame(story)
for case in CASES:
    print(f"\n=== {case} ===")
    print(STORY[STORY["case"] == case].drop(columns="case").to_string(index=False, float_format=fmt))


=== lin_all ===
    dataset                               model     mu*       u       A       B  B-A mean           B-A CI B>A  B - best mean-only best
  10x_TuPro                     heteroscedastic +0.1971 +0.2144 +0.2241 +0.2392   +0.0175 [+0.005, +0.030] 7/7             +0.0151    B
  10x_TuPro                  spot_homoscedastic +0.1907 +0.1204 +0.2359 +0.2492   +0.0149 [+0.003, +0.026] 6/7             +0.0133    B
  10x_TuPro negativebinomial_hetero_noanchor_cp +0.3493 +0.3262 +0.2968 +0.3052   +0.0118 [+0.001, +0.022] 6/7             -0.0441  mu*
        TLS                     heteroscedastic +0.2798 +0.1278 +0.2207 +0.2226   +0.0015 [-0.004, +0.007] 4/6             -0.0572  mu*
        TLS                  spot_homoscedastic +0.2631 -0.0655 +0.2588 +0.2567   -0.0015 [-0.009, +0.007] 2/6             -0.0064  mu*
        TLS negativebinomial_hetero_noanchor_cp +0.2381 +0.1731 +0.1680 +0.1674   -0.0014 [-0.033, +0.030] 4/6             -0.0707  mu*
LUNG_CANCER                    